# CondVQA - Conditional Visual Question Answering with Integrated Gradients

**Requirements:** Google Colab with GPU runtime (Runtime > Change runtime type > GPU)

## Section 1: Setup

In [ ]:
# Install numpy (required specific version)
!pip uninstall -y numpy
!pip install numpy==1.26.4

print("\n!! Please restart runtime: Runtime > Restart runtime !!")
print("After restart, skip this cell and run the next one.")

In [ ]:
# Clone repo and install dependencies
!git clone -b clean https://github.com/rdgbrandon/CondVQA.git
%cd CondVQA
!pip install -q -r requirements.txt

In [ ]:
# Load models
import sys
sys.path.insert(0, './src')

from src import load_model, vqa_interpret, conditional_query_vqa_interpret, text_vqa_interpret
import torch
import gc

model, processor = load_model()

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("WARNING: GPU not available. Enable GPU in Runtime > Change runtime type")

## Section 2: Image Analysis

Upload an image, run VQA with Integrated Gradients attribution heatmaps and text attribution.

In [ ]:
# Upload image
gc.collect()
torch.cuda.empty_cache()

from google.colab import files
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
print(f"Uploaded: {image_path}")

In [ ]:
# Image VQA with Integrated Gradients attribution
questions = [
    "What is in this image?",
    "What color is the main object?"
]

vqa_interpret(
    image_path=image_path,
    questions=questions,
    model=model,
    processor=processor,
    show_top_k=10
)

In [ ]:
# Text attribution - which words in the question matter most?
text_results = text_vqa_interpret(
    image_path=image_path,
    questions=questions,
    model=model,
    processor=processor,
    mode='both',
    n_steps=10,
    show_visualizations=True
)

## Section 3: Batch Test (200 Test Cases)

Runs all 200 test cases from `test/CondVQA_3cols.csv` against their expected answers and outputs a comparison summary.

In [ ]:
# Set up paths to test folder (inside cloned repo)
import os
import sys

test_dir = os.path.join(os.getcwd(), 'test')
sys.path.insert(0, test_dir)

from run_batch_test import run_batch_test

gc.collect()
torch.cuda.empty_cache()

csv_path = os.path.join(test_dir, 'CondVQA_3cols.csv')
video_dir = os.path.join(test_dir, 'testcases')

print(f"CSV path: {csv_path}")
print(f"Video dir: {video_dir}")
print(f"CSV exists: {os.path.exists(csv_path)}")
print(f"Video dir exists: {os.path.exists(video_dir)}")

In [ ]:
# Run all 200 test cases
results = run_batch_test(
    csv_path=csv_path,
    video_dir=video_dir,
    model=model,
    processor=processor,
    fps_sample=1,
    confidence_threshold=0.5,
    aggregation_method='most_confident',
    show_visualizations=False,
    save_results=True,
    output_dir=os.path.join(test_dir, 'batch_results')
)